In [1]:
import pandas as pd
from app.transport.osm_tools import geocode_address, get_distance_km

# Load data
employees = pd.read_csv("data/processed/employee.csv")
facilities = pd.read_csv("data/processed/facil.csv")

# Merge to get facility address
df = employees.merge(
    facilities[["facility_id", "address"]],
    on="facility_id",
    how="left"
)

distances = []

for _, row in df.iterrows():
    home_lat, home_lon = geocode_address(row["home_address"])
    work_lat, work_lon = geocode_address(row["address"])

    distance = get_distance_km(home_lat, home_lon, work_lat, work_lon)

    distances.append(distance)

df["distance_km"] = distances
EMISSION_FACTORS = {
    "car": 0.21,
    "bus": 0.10,
    "train": 0.04,
    "bike": 0.0,
    "walking": 0.0
}

df["emission_kgco2_per_day"] = df.apply(
    lambda row: row["distance_km"] * EMISSION_FACTORS[row["transport_mode"]],
    axis=1
)

df["annual_emission_kgco2"] = df["emission_kgco2_per_day"] * 220
df.to_csv("data/processed/employee_transport.csv", index=False)

In [10]:
df.to_csv("data/processed/employee_transport.csv", index=False)